# 論文・レポート用図生成

この notebook は、既存の解析済み CSV / JSON から論文・レポート掲載用の図を生成します。元 pcap や大規模 flow CSV の再処理は行わず、`results/comparison`、`results/prefix`、`results/features` の summary を読みます。

出力: `OUTPUT_DIR` に PDF / PNG などを保存します。`OVERWRITE=False` の場合、既存ファイルを上書きせず連番付きファイル名で保存します。

注意: 検出区間や比較区間は notebook 冒頭の設定で指定します。検出区間は図上の補助表示であり、単独で異常を断定するものではありません。

In [ ]:
# 設定セル: 図生成前にここだけ変更してください。
DATASET = "202604081300"       # 単一 dataset 図の対象。None の場合は自動選択。
DATASETS_FOR_TIMESERIES = None  # 例: ["202604081200", "202604081300", "202604081400"]。None で全 comparison summary。
TARGET_PATTERN = None           # 例: "dst_157.209.123.224_27"。None なら各図で non-overall を使用。
ANALYSIS_MODE = "comparison"    # 現状は comparison / prefix_summary の既存 CSV を想定。
OUTPUT_DIR = None               # None なら results/reports/figures/<dataset>/
FORMATS = ["pdf", "png"]  # 論文用に PDF と確認用 PNG の両方を保存します。
OVERWRITE = False

DISPLAY_RANGE = None            # 例: ("202604081200", "202604081400")。dataset 名の範囲で絞り込み。
DETECTION_INTERVALS = ["202604081300"]  # 時系列図で強調したい dataset 名。

PRIMARY_METRIC = "byte_count_mean"
SECONDARY_METRIC = "duration_mean"
SCATTER_X = "flow_count"
SCATTER_Y = "byte_count_mean"
SCATTER_COLOR = "short_flow_ratio"
DELTA_METRICS = [
    "short_flow_ratio",
    "tiny_flow_ratio",
    "rst_observed_flow_ratio",
    "tcp_flow_ratio",
    "udp_flow_ratio",
    "duration_mean",
    "packet_count_mean",
    "byte_count_mean",
]

In [ ]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Image, display

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.size"] = 10
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("project root not found: expected AGENTS.md and results/")


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


def rel(path: Path | None) -> str:
    if path is None:
        return ""
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def read_csv(path: Path | None) -> pd.DataFrame:
    if path is None or not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json(path: Path | None) -> dict[str, Any]:
    if path is None or not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    return data if isinstance(data, dict) else {}


def available_datasets() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    sources = {
        "comparison": RESULTS_DIR / "comparison",
        "prefix": RESULTS_DIR / "prefix",
        "features_all": RESULTS_DIR / "features" / "all",
        "aguri": RESULTS_DIR / "aguri",
        "reports": RESULTS_DIR / "reports",
    }
    names = sorted({p.name for base in sources.values() if base.exists() for p in base.iterdir() if p.is_dir()})
    for name in names:
        rows.append({
            "dataset": name,
            "comparison_summary": (RESULTS_DIR / "comparison" / name / "comparison_summary.csv").exists(),
            "prefix_evaluation": (RESULTS_DIR / "prefix" / name / "prefix_evaluation.csv").exists(),
            "selected_prefixes": (RESULTS_DIR / "prefix" / name / "selected_prefixes.csv").exists(),
            "overall_features": (RESULTS_DIR / "features" / "all" / name / "features.json").exists(),
            "aguri_candidates": (RESULTS_DIR / "aguri" / name / f"{name}.aguri_candidates.csv").exists(),
            "report_html": (RESULTS_DIR / "reports" / name / "analysis_report.html").exists(),
        })
    return pd.DataFrame(rows)


def choose_dataset(configured: str | None) -> str:
    datasets = available_datasets()
    if datasets.empty:
        raise FileNotFoundError("no result datasets found under results/")
    if configured:
        if configured not in set(datasets["dataset"]):
            print(f"[warn] configured dataset not discovered: {configured}")
        return configured
    scored = datasets.assign(score=datasets.drop(columns=["dataset"]).sum(axis=1))
    return str(scored.sort_values(["comparison_summary", "prefix_evaluation", "score", "dataset"], ascending=[False, False, False, True]).iloc[0]["dataset"])


def dataset_paths(dataset: str) -> dict[str, Path]:
    return {
        "comparison_summary": RESULTS_DIR / "comparison" / dataset / "comparison_summary.csv",
        "comparison_plots": RESULTS_DIR / "comparison" / dataset / "plots",
        "prefix_evaluation": RESULTS_DIR / "prefix" / dataset / "prefix_evaluation.csv",
        "selected_prefixes": RESULTS_DIR / "prefix" / dataset / "selected_prefixes.csv",
        "overall_features": RESULTS_DIR / "features" / "all" / dataset / "features.json",
        "flow_plots_all": RESULTS_DIR / "flow_plots" / "all" / dataset,
        "flow_plots_prefix": RESULTS_DIR / "flow_plots" / "prefix" / dataset,
        "aguri_candidates": RESULTS_DIR / "aguri" / dataset / f"{dataset}.aguri_candidates.csv",
        "report_html": RESULTS_DIR / "reports" / dataset / "analysis_report.html",
    }


def file_audit(paths: dict[str, Path]) -> pd.DataFrame:
    rows = []
    for key, path in paths.items():
        if path.is_dir():
            count = len([p for p in path.rglob("*") if p.is_file()])
            rows.append({"name": key, "path": rel(path), "exists": True, "type": "dir", "items": count, "rows": None, "columns": None})
        elif path.exists():
            rows.append({"name": key, "path": rel(path), "exists": True, "type": path.suffix.lstrip("."), "items": None, "rows": safe_row_count(path), "columns": safe_columns(path)})
        else:
            rows.append({"name": key, "path": rel(path), "exists": False, "type": "", "items": None, "rows": None, "columns": None})
    return pd.DataFrame(rows)


def safe_row_count(path: Path) -> int | None:
    if path.suffix.lower() != ".csv":
        return None
    try:
        return len(pd.read_csv(path))
    except Exception:
        return None


def safe_columns(path: Path) -> str | None:
    try:
        if path.suffix.lower() == ".csv":
            return ", ".join(pd.read_csv(path, nrows=0).columns)
        if path.suffix.lower() == ".json":
            return ", ".join(read_json(path).keys())
    except Exception:
        return None
    return None


def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "missing_ratio": [float(df[c].isna().mean()) for c in df.columns],
    })


def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lowered = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    return None


def numeric_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    col = first_existing_column(df, candidates)
    if col is None:
        return None
    converted = pd.to_numeric(df[col], errors="coerce")
    if converted.notna().any():
        df[col] = converted
        return col
    return None


def target_col(df: pd.DataFrame) -> str | None:
    return first_existing_column(df, ["target", "normalized_dst_prefix", "dst_prefix", "prefix", "aggregate_id"])


def sanitize_target(name: str) -> str:
    return "".join(c if c.isalnum() or c in ("-", "_", ".") else "_" for c in str(name))


def target_candidates(comparison: pd.DataFrame, selected: pd.DataFrame, evaluation: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for source, df in [("comparison_summary", comparison), ("selected_prefixes", selected), ("prefix_evaluation", evaluation)]:
        col = target_col(df)
        if col is None:
            continue
        for _, row in df.iterrows():
            value = str(row[col])
            if value.lower() == "overall":
                continue
            rows.append({"source": source, "target": value, "plot_dir_name": sanitize_target(value), **{k: row[k] for k in df.columns if k in {"flow_count", "packet_count", "byte_count", "score", "scan_candidate", "passes_filters"}}})
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.drop_duplicates(["source", "target"]).reset_index(drop=True)


def choose_target(configured: str | None, candidates: pd.DataFrame) -> str | None:
    if configured:
        return configured
    if candidates.empty:
        return None
    for source in ["selected_prefixes", "comparison_summary", "prefix_evaluation"]:
        sub = candidates[candidates["source"] == source]
        if not sub.empty:
            return str(sub.iloc[0]["target"])
    return str(candidates.iloc[0]["target"])


def find_target_plot_dir(plots_root: Path, target: str | None) -> Path | None:
    if target is None or not plots_root.exists():
        return None
    names = [sanitize_target(target)]
    if "/" in str(target):
        names.append(sanitize_target(str(target).replace("/", "_")))
    existing = {p.name: p for p in plots_root.iterdir() if p.is_dir()}
    for name in names:
        if name in existing:
            return existing[name]
    target_norm = sanitize_target(target).lower()
    for name, path in existing.items():
        if name.lower() == target_norm or target_norm in name.lower() or name.lower() in target_norm:
            return path
    return None


def image_grid(paths: list[Path], max_width: int = 720) -> None:
    if not paths:
        print("no images found")
        return
    for path in paths:
        display(HTML(f"<div style='font-weight:600;margin-top:12px'>{rel(path)}</div>"))
        display(Image(filename=str(path), width=max_width))


def feature_rows(features_json: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for name, block in features_json.get("features", {}).items():
        stats = block.get("stats", {}) if isinstance(block, dict) else {}
        rows.append({"feature": name, "unit": block.get("unit", ""), "log_scale_recommended": block.get("log_scale_recommended", False), **stats})
    return pd.DataFrame(rows)


def histogram_block(features_json: dict[str, Any], feature: str, prefer_log: bool = True) -> tuple[str, dict[str, Any], dict[str, Any]] | None:
    block = features_json.get("features", {}).get(feature)
    if not isinstance(block, dict):
        return None
    if prefer_log and block.get("log_scale_recommended") and isinstance(block.get("log_histogram"), dict):
        return "log_histogram", block["log_histogram"], block
    if isinstance(block.get("histogram"), dict):
        return "histogram", block["histogram"], block
    if isinstance(block.get("log_histogram"), dict):
        return "log_histogram", block["log_histogram"], block
    return None


def hist_edges(hist_type: str, hist: dict[str, Any]) -> list[float] | None:
    raw = hist.get("linear_edges") if hist_type == "log_histogram" else hist.get("edges")
    if raw is None:
        raw = hist.get("bins") if isinstance(hist.get("bins"), list) else None
    if not isinstance(raw, list):
        return None
    try:
        return [float(v) for v in raw]
    except Exception:
        return None


def hist_counts(hist: dict[str, Any]) -> list[float] | None:
    raw = hist.get("counts")
    if not isinstance(raw, list):
        return None
    try:
        return [float(v) for v in raw]
    except Exception:
        return None


def plot_hist_and_cdf(features_json: dict[str, Any], feature_names: list[str], title_prefix: str = "") -> None:
    available = [f for f in feature_names if histogram_block(features_json, f) is not None]
    if not available:
        print("no plottable histogram features found")
        return
    fig, axes = plt.subplots(len(available), 2, figsize=(12, max(3.2, 2.8 * len(available))))
    axes = np.atleast_2d(axes)
    for row, feature in enumerate(available):
        hist_type, hist, block = histogram_block(features_json, feature)
        edges = hist_edges(hist_type, hist)
        counts = hist_counts(hist)
        if edges is None or counts is None or len(edges) != len(counts) + 1:
            continue
        widths = np.diff(edges)
        axes[row, 0].bar(edges[:-1], counts, width=widths, align="edge", color="#4C78A8", edgecolor="white", linewidth=0.4)
        if hist_type == "log_histogram":
            axes[row, 0].set_xscale("log")
        axes[row, 0].set_title(f"{feature} histogram")
        axes[row, 0].set_ylabel("flow count")
        total = np.sum(counts)
        cdf = np.cumsum(counts) / total if total else np.zeros(len(counts))
        axes[row, 1].step(edges[1:], cdf, where="post", color="#F58518")
        if hist_type == "log_histogram":
            axes[row, 1].set_xscale("log")
        axes[row, 1].set_ylim(0, 1.02)
        axes[row, 1].set_title(f"{feature} CDF")
        axes[row, 1].set_ylabel("cumulative ratio")
    if title_prefix:
        fig.suptitle(title_prefix, y=1.01, fontsize=12)
    fig.tight_layout()
    plt.show()


def plot_topk(df: pd.DataFrame, metric: str, label_col: str | None = None, top_k: int = 15, title: str = "") -> None:
    if df.empty or metric not in df.columns:
        print(f"skip top-k: missing {metric}")
        return
    label_col = label_col or target_col(df)
    if label_col is None:
        print("skip top-k: no label column")
        return
    work = df.copy()
    work[metric] = pd.to_numeric(work[metric], errors="coerce")
    work = work.dropna(subset=[metric]).sort_values(metric, ascending=False).head(top_k)
    if work.empty:
        print(f"skip top-k: no numeric values for {metric}")
        return
    fig, ax = plt.subplots(figsize=(9, max(3.0, 0.36 * len(work) + 1.0)))
    ax.barh(work[label_col].astype(str)[::-1], work[metric][::-1], color="#4C78A8", edgecolor="white")
    ax.set_xlabel(metric)
    ax.set_title(title or f"Top {len(work)} by {metric}")
    fig.tight_layout()
    plt.show()


PUBLICATION_STYLE = {
    "figure.figsize": (7.2, 4.2),
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
}
plt.rcParams.update(PUBLICATION_STYLE)


def ensure_output_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def output_path(outdir: Path, stem: str, ext: str, overwrite: bool) -> Path:
    ext = ext.lstrip(".")
    base = outdir / f"{stem}.{ext}"
    if overwrite or not base.exists():
        return base
    for i in range(1, 1000):
        candidate = outdir / f"{stem}_{i:02d}.{ext}"
        if not candidate.exists():
            return candidate
    raise FileExistsError(f"too many existing files for {base}")


def save_current(fig: plt.Figure, outdir: Path, stem: str, formats: list[str], overwrite: bool) -> list[Path]:
    saved = []
    for fmt in formats:
        path = output_path(outdir, stem, fmt, overwrite)
        fig.savefig(path, bbox_inches="tight")
        saved.append(path)
    return saved


def plot_metric_timeseries(summary_by_dataset: pd.DataFrame, metric: str, target_pattern: str | None, outdir: Path, formats: list[str], overwrite: bool, highlight_datasets: list[str] | None = None) -> list[Path]:
    if summary_by_dataset.empty or metric not in summary_by_dataset.columns:
        print(f"skip timeseries: missing {metric}")
        return []
    target_col_name = target_col(summary_by_dataset)
    if target_col_name is None:
        print("skip timeseries: no target column")
        return []
    work = summary_by_dataset.copy()
    if target_pattern:
        mask = work[target_col_name].astype(str).str.contains(target_pattern, regex=False, na=False)
        selected = work[mask]
        if selected.empty:
            print(f"target pattern not found, using non-overall rows: {target_pattern}")
            selected = work[work[target_col_name].astype(str).str.lower() != "overall"]
    else:
        selected = work[work[target_col_name].astype(str).str.lower() != "overall"]
    overall = work[work[target_col_name].astype(str).str.lower() == "overall"]
    selected[metric] = pd.to_numeric(selected[metric], errors="coerce")
    overall[metric] = pd.to_numeric(overall[metric], errors="coerce")
    fig, ax = plt.subplots()
    if not overall.empty:
        ax.plot(overall["dataset"], overall[metric], marker="o", linewidth=1.8, label="overall", color="#4C78A8")
    if not selected.empty:
        agg = selected.groupby("dataset", as_index=False)[metric].median()
        ax.plot(agg["dataset"], agg[metric], marker="s", linewidth=1.8, label="target/prefix median", color="#F58518")
    for label in highlight_datasets or []:
        if label in set(work["dataset"].astype(str)):
            ax.axvline(label, color="#E45756", alpha=0.55, linewidth=1.2, linestyle="--", label="detection/comparison interval" if "detection/comparison interval" not in ax.get_legend_handles_labels()[1] else None)
    ax.set_xlabel("dataset")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} across datasets")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(frameon=False)
    fig.tight_layout()
    saved = save_current(fig, outdir, f"timeseries_{metric}", formats, overwrite)
    plt.show()
    return saved


def plot_overall_vs_targets(df: pd.DataFrame, x_metric: str, y_metric: str, outdir: Path, formats: list[str], overwrite: bool, color_metric: str | None = None, title: str = "") -> list[Path]:
    if df.empty or x_metric not in df.columns or y_metric not in df.columns:
        print(f"skip scatter: missing {x_metric} or {y_metric}")
        return []
    tcol = target_col(df)
    if tcol is None:
        print("skip scatter: no target column")
        return []
    work = df.copy()
    work[x_metric] = pd.to_numeric(work[x_metric], errors="coerce")
    work[y_metric] = pd.to_numeric(work[y_metric], errors="coerce")
    work = work.dropna(subset=[x_metric, y_metric])
    fig, ax = plt.subplots()
    non_overall = work[work[tcol].astype(str).str.lower() != "overall"]
    overall = work[work[tcol].astype(str).str.lower() == "overall"]
    if color_metric and color_metric in non_overall.columns:
        c = pd.to_numeric(non_overall[color_metric], errors="coerce")
        sc = ax.scatter(non_overall[x_metric], non_overall[y_metric], c=c, cmap="viridis", s=44, alpha=0.82, edgecolor="white", linewidth=0.4, label="prefix")
        fig.colorbar(sc, ax=ax, label=color_metric)
    else:
        ax.scatter(non_overall[x_metric], non_overall[y_metric], color="#4C78A8", s=44, alpha=0.82, edgecolor="white", linewidth=0.4, label="prefix")
    if not overall.empty:
        ax.scatter(overall[x_metric], overall[y_metric], color="#E45756", marker="*", s=180, label="overall", zorder=5)
    ax.set_xlabel(x_metric)
    ax.set_ylabel(y_metric)
    ax.set_title(title or f"{y_metric} vs {x_metric}")
    ax.legend(frameon=False)
    fig.tight_layout()
    saved = save_current(fig, outdir, f"scatter_{x_metric}_vs_{y_metric}", formats, overwrite)
    plt.show()
    return saved


def plot_baseline_delta(df: pd.DataFrame, metrics: list[str], outdir: Path, formats: list[str], overwrite: bool, target_pattern: str | None = None) -> list[Path]:
    if df.empty:
        return []
    tcol = target_col(df)
    if tcol is None:
        print("skip delta: no target column")
        return []
    overall = df[df[tcol].astype(str).str.lower() == "overall"]
    targets = df[df[tcol].astype(str).str.lower() != "overall"]
    if target_pattern:
        targets = targets[targets[tcol].astype(str).str.contains(target_pattern, regex=False, na=False)]
    if overall.empty or targets.empty:
        print("skip delta: overall or target rows missing")
        return []
    baseline = overall.iloc[0]
    target = targets.iloc[0]
    rows = []
    for metric in metrics:
        if metric not in df.columns:
            continue
        b = pd.to_numeric(pd.Series([baseline[metric]]), errors="coerce").iloc[0]
        t = pd.to_numeric(pd.Series([target[metric]]), errors="coerce").iloc[0]
        if pd.notna(b) and pd.notna(t):
            rows.append({"metric": metric, "delta": t - b, "ratio": t / b if b != 0 else np.nan})
    work = pd.DataFrame(rows)
    if work.empty:
        print("skip delta: no numeric metric pairs")
        return []
    fig, ax = plt.subplots(figsize=(7.2, max(3.0, 0.38 * len(work) + 1.0)))
    ax.barh(work["metric"][::-1], work["delta"][::-1], color="#F58518", edgecolor="white")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("target minus overall")
    ax.set_title("Difference from overall baseline")
    fig.tight_layout()
    saved = save_current(fig, outdir, "baseline_delta", formats, overwrite)
    plt.show()
    display(work)
    return saved


def load_all_comparison_summaries(selected_datasets: list[str] | None = None) -> pd.DataFrame:
    rows = []
    base = RESULTS_DIR / "comparison"
    for summary in sorted(base.glob("*/comparison_summary.csv")) if base.exists() else []:
        dataset = summary.parent.name
        if selected_datasets and dataset not in selected_datasets:
            continue
        df = pd.read_csv(summary)
        df.insert(0, "dataset", dataset)
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

In [ ]:
datasets = available_datasets()
display(datasets)

dataset = choose_dataset(DATASET)
paths = dataset_paths(dataset)
outdir = ensure_output_dir(Path(OUTPUT_DIR).expanduser() if OUTPUT_DIR else RESULTS_DIR / "reports" / "figures" / dataset)
print(f"selected dataset = {dataset}")
print(f"output directory = {outdir}")
display(file_audit(paths))

## 入力 CSV の監査

図生成の前に、読み込んだファイル、行数、列名、欠損を確認します。列名の揺れは候補列から自動検出しますが、意図と違う列が選ばれていないかここで確認してください。

In [ ]:
comparison = read_csv(paths["comparison_summary"])
prefix_eval = read_csv(paths["prefix_evaluation"])
selected = read_csv(paths["selected_prefixes"])
overall_features = read_json(paths["overall_features"])

for name, df in [("comparison_summary", comparison), ("prefix_evaluation", prefix_eval), ("selected_prefixes", selected)]:
    print(f"\n{name}: rows={len(df)}, columns={list(df.columns)}")
    if not df.empty:
        display(missing_summary(df).head(100))

totals = overall_features.get("totals", {})
if totals:
    display(pd.DataFrame([totals]))

## Dataset 間の時系列図

複数 dataset の `comparison_summary.csv` を読み込み、overall と対象 prefix 群の指標を dataset 軸で比較します。`DETECTION_INTERVALS` に含めた dataset は薄く強調表示します。

In [ ]:
all_summaries = load_all_comparison_summaries(DATASETS_FOR_TIMESERIES)
if DISPLAY_RANGE and not all_summaries.empty:
    start, end = DISPLAY_RANGE
    all_summaries = all_summaries[(all_summaries["dataset"].astype(str) >= start) & (all_summaries["dataset"].astype(str) <= end)]
print(f"all comparison rows = {len(all_summaries)}")
display(all_summaries.head(30))

saved = []
saved += plot_metric_timeseries(all_summaries, PRIMARY_METRIC, TARGET_PATTERN, outdir, FORMATS, OVERWRITE, DETECTION_INTERVALS)
saved += plot_metric_timeseries(all_summaries, SECONDARY_METRIC, TARGET_PATTERN, outdir, FORMATS, OVERWRITE, DETECTION_INTERVALS)
print("saved:")
for path in saved:
    print(" -", rel(path))

## 対象全体の中での位置づけ

単一 dataset の `comparison_summary.csv` 上で、overall と各 prefix を散布図として表示します。色には scan 的傾向や短命 flow 比率など、解釈に使う指標を指定できます。

In [ ]:
saved = []
saved += plot_overall_vs_targets(comparison, SCATTER_X, SCATTER_Y, outdir, FORMATS, OVERWRITE, color_metric=SCATTER_COLOR)
print("saved:")
for path in saved:
    print(" -", rel(path))

## Baseline との差分図

overall 行を baseline とし、対象 prefix の各指標との差分を図化します。短命 flow や RST などは、異常の断定ではなく通信特性の差として扱います。

In [ ]:
saved = plot_baseline_delta(comparison, DELTA_METRICS, outdir, FORMATS, OVERWRITE, TARGET_PATTERN)
print("saved:")
for path in saved:
    print(" -", rel(path))

## Prefix 評価 CSV からのレポート向け図

`prefix_evaluation.csv` がある場合、score、flow_count、byte_count、scan_candidate などを使って、選定候補の分布やランキングを図化します。

In [ ]:
if prefix_eval.empty:
    print("prefix_evaluation.csv not found")
else:
    for metric in ["score", "flow_count", "byte_count", "short_flow_ratio", "tiny_flow_ratio", "rst_observed_ratio"]:
        if metric in prefix_eval.columns:
            plot_topk(prefix_eval, metric, top_k=15, title=f"Prefix ranking by {metric}")
            fig = plt.gcf()
            # plot_topk displays and closes nothing; generate a dedicated saved version here for stable files.
            work = prefix_eval.copy()
            label = target_col(work)
            work[metric] = pd.to_numeric(work[metric], errors="coerce")
            work = work.dropna(subset=[metric]).sort_values(metric, ascending=False).head(15)
            if label and not work.empty:
                fig, ax = plt.subplots(figsize=(7.2, max(3.0, 0.36 * len(work) + 1.0)))
                ax.barh(work[label].astype(str)[::-1], work[metric][::-1], color="#4C78A8", edgecolor="white")
                ax.set_xlabel(metric)
                ax.set_title(f"Prefix ranking by {metric}")
                fig.tight_layout()
                saved = save_current(fig, outdir, f"prefix_rank_{metric}", FORMATS, OVERWRITE)
                plt.show()
                for path in saved:
                    print(" -", rel(path))

    x = numeric_col(prefix_eval, ["flow_count"])
    y = numeric_col(prefix_eval, ["byte_count"])
    c = numeric_col(prefix_eval, ["score"])
    if x and y:
        fig, ax = plt.subplots()
        if c:
            sc = ax.scatter(prefix_eval[x], prefix_eval[y], c=prefix_eval[c], cmap="viridis", s=46, alpha=0.82, edgecolor="white", linewidth=0.4)
            fig.colorbar(sc, ax=ax, label=c)
        else:
            ax.scatter(prefix_eval[x], prefix_eval[y], color="#4C78A8", s=46, alpha=0.82, edgecolor="white", linewidth=0.4)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(x)
        ax.set_ylabel(y)
        ax.set_title("Prefix scale and score")
        fig.tight_layout()
        saved = save_current(fig, outdir, "prefix_scale_score", FORMATS, OVERWRITE)
        plt.show()
        for path in saved:
            print(" -", rel(path))

## 保存ファイル一覧

生成した図を確認します。`OVERWRITE=False` の場合、同名ファイルがあると連番付きで保存されます。

In [ ]:
created = sorted([p for p in outdir.glob("*") if p.suffix.lower().lstrip(".") in set(FORMATS)])
print(f"created/available figures in main output dir: {len(created)}")
for path in created:
    print(rel(path))

if "PAPER_FIGURE_OUTPUT_DIR" in globals():
    paper_created = sorted([p for p in PAPER_FIGURE_OUTPUT_DIR.glob("*") if p.suffix.lower().lstrip(".") in set(PAPER_FIGURE_FORMATS)])
    print(f"created/available figures in paper figure dir: {len(paper_created)}")
    for path in paper_created:
        print(rel(path))

## 予稿用: 全体トラフィックと prefix 中央値の比較

ソサイエティ大会 1 ページ予稿向けに、`overall` と複数 prefix の `packet_count_median` / `byte_count_median` を比較する 2 パネル図を PDF / PNG の両形式で生成します。低 flow 数の prefix は中央値が不安定になりやすいため、`MIN_PREFIX_FLOW_COUNT` で除外します。


In [ ]:
# 予稿用図の設定: 必要に応じてここだけ変更してください。
PAPER_FIGURE_DATASET = dataset if "dataset" in globals() else choose_dataset(DATASET)
PAPER_FIGURE_OUTPUT_DIR = RESULTS_DIR / "paper_figures" / PAPER_FIGURE_DATASET

MIN_PREFIX_FLOW_COUNT = 1000
MAX_PREFIXES_TO_PLOT = 8
SORT_BY = "flow_count"  # "flow_count" or "byte_count_median"
PLOT_LOG_SCALE_VERSION = True

PAPER_FIGURE_FORMATS = FORMATS if "FORMATS" in globals() else ["pdf", "png"]
PAPER_FIGURE_OVERWRITE = OVERWRITE if "OVERWRITE" in globals() else False

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


def normalize_prefix_label(label: str) -> str:
    """Convert internal target names such as dst_202.244.127.0_24 to CIDR-like labels."""
    text = str(label).strip()
    if text.lower() == "overall":
        return "overall"
    text = re.sub(r"^(src|dst)_", "", text)
    if "/" in text:
        return text
    sanitized_ipv6_match = re.fullmatch(r"([0-9A-Fa-f_]+)___(\d{1,3})", text)
    if sanitized_ipv6_match:
        addr = sanitized_ipv6_match.group(1).replace("_", ":") + "::"
        return f"{addr}/{sanitized_ipv6_match.group(2)}"
    ipv4_match = re.fullmatch(r"((?:\d{1,3}\.){3}\d{1,3})_(\d{1,2})", text)
    if ipv4_match:
        return f"{ipv4_match.group(1)}/{ipv4_match.group(2)}"
    ipv6_match = re.fullmatch(r"([0-9A-Fa-f:]+)_(\d{1,3})", text)
    if ipv6_match and ":" in ipv6_match.group(1):
        return f"{ipv6_match.group(1)}/{ipv6_match.group(2)}"
    return text


def _comparison_summary_path(dataset: str) -> Path:
    return RESULTS_DIR / "comparison" / dataset / "comparison_summary.csv"


def _coerce_summary_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize summary columns used by the paper figure while preserving other columns."""
    if df.empty:
        return df.copy()
    out = df.copy()
    rename_map = {
        "median_duration": "duration_median",
        "median_packet_count": "packet_count_median",
        "median_byte_count": "byte_count_median",
        "median_avg_packet_size": "avg_packet_size_median",
    }
    for old, new in rename_map.items():
        if old in out.columns and new not in out.columns:
            out[new] = out[old]
    for col in [
        "flow_count",
        "duration_median",
        "packet_count_median",
        "byte_count_median",
        "avg_packet_size_median",
        "tcp_ratio",
        "udp_ratio",
        "tcp_flow_ratio",
        "udp_flow_ratio",
    ]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def load_overall_summary(dataset: str) -> pd.DataFrame:
    """Load the overall row from comparison_summary.csv for a dataset."""
    path = _comparison_summary_path(dataset)
    if not path.exists():
        print(f"[warn] overall summary source not found: {rel(path)}")
        return pd.DataFrame()
    df = _coerce_summary_columns(pd.read_csv(path))
    tcol = target_col(df)
    if tcol is None:
        print(f"[warn] target column not found in {rel(path)}")
        return pd.DataFrame()
    overall = df[df[tcol].astype(str).str.lower() == "overall"].copy()
    if overall.empty:
        print(f"[warn] overall row not found in {rel(path)}")
        return pd.DataFrame()
    overall["display_label"] = "overall"
    return overall.reset_index(drop=True)


def load_prefix_summaries(dataset: str, min_flow_count: int = 1000) -> pd.DataFrame:
    """Load prefix rows from comparison_summary.csv and apply a configurable flow-count filter."""
    path = _comparison_summary_path(dataset)
    if not path.exists():
        print(f"[warn] prefix summary source not found: {rel(path)}")
        return pd.DataFrame()
    df = _coerce_summary_columns(pd.read_csv(path))
    tcol = target_col(df)
    if tcol is None:
        print(f"[warn] target column not found in {rel(path)}")
        return pd.DataFrame()
    prefix_df = df[df[tcol].astype(str).str.lower() != "overall"].copy()
    prefix_df = prefix_df.rename(columns={tcol: "prefix"})
    if "flow_count" not in prefix_df.columns:
        print(f"[warn] flow_count column not found in {rel(path)}; no flow-count filter applied")
    else:
        prefix_df = prefix_df[prefix_df["flow_count"].fillna(0) >= min_flow_count]
    prefix_df["display_label"] = prefix_df["prefix"].map(normalize_prefix_label)
    return prefix_df.reset_index(drop=True)


def _required_median_columns(df: pd.DataFrame, metrics: list[str]) -> list[str]:
    return [metric for metric in metrics if metric in df.columns and pd.to_numeric(df[metric], errors="coerce").notna().any()]


def _save_paper_figure(fig: plt.Figure, output_dir: Path, stem: str, formats: list[str], overwrite: bool) -> list[Path]:
    ensure_output_dir(output_dir)
    saved_paths = save_current(fig, output_dir, stem, formats, overwrite)
    for saved_path in saved_paths:
        print("saved:", rel(saved_path))
    return saved_paths


def plot_prefix_median_comparison(
    overall_df: pd.DataFrame,
    prefix_df: pd.DataFrame,
    output_dir: Path,
    max_prefixes: int = 8,
    sort_by: str = "flow_count",
    use_log_y: bool = False,
) -> None:
    """Plot packet and byte median comparison with overall fixed at the left edge."""
    metrics = _required_median_columns(
        pd.concat([overall_df, prefix_df], ignore_index=True, sort=False),
        ["packet_count_median", "byte_count_median"],
    )
    if not metrics:
        print("[warn] no packet/byte median columns available; skip comparison figure")
        return
    if overall_df.empty or prefix_df.empty:
        print("[warn] overall or prefix summary is empty; skip comparison figure")
        return

    work_prefix = prefix_df.copy()
    if sort_by not in work_prefix.columns:
        print(f"[warn] SORT_BY={sort_by!r} not found; falling back to flow_count")
        sort_by = "flow_count" if "flow_count" in work_prefix.columns else metrics[0]
    work_prefix[sort_by] = pd.to_numeric(work_prefix[sort_by], errors="coerce")
    work_prefix = work_prefix.sort_values(sort_by, ascending=False).head(max_prefixes)

    overall_row = overall_df.iloc[[0]].copy()
    if "prefix" not in overall_row.columns:
        overall_row["prefix"] = "overall"
    overall_row["display_label"] = "overall"
    plot_df = pd.concat([overall_row, work_prefix], ignore_index=True, sort=False)
    plot_df["display_label"] = plot_df["display_label"].fillna(plot_df.get("prefix", "").astype(str).map(normalize_prefix_label))

    fig, axes = plt.subplots(1, len(metrics), figsize=(7.0, 3.1), sharex=False)
    axes = np.atleast_1d(axes)
    colors = ["#5B5B5B"] + ["#4C78A8"] * (len(plot_df) - 1)
    x = np.arange(len(plot_df))

    ylabels = {
        "packet_count_median": "Median packets per flow",
        "byte_count_median": "Median bytes per flow",
    }
    titles = {
        "packet_count_median": "Packet count",
        "byte_count_median": "Byte count",
    }
    for ax, metric in zip(axes, metrics):
        values = pd.to_numeric(plot_df[metric], errors="coerce")
        ax.bar(x, values, color=colors, edgecolor="white", linewidth=0.6)
        ax.set_title(titles.get(metric, metric))
        ax.set_ylabel(ylabels.get(metric, metric))
        ax.set_xticks(x)
        ax.set_xticklabels(plot_df["display_label"], rotation=35, ha="right")
        if use_log_y:
            positive = values[values > 0]
            if not positive.empty:
                ax.set_yscale("log")
                ax.set_ylabel(f"{ylabels.get(metric, metric)} (log scale)")
        ax.margins(x=0.02)
    fig.tight_layout(w_pad=1.5)
    suffix = "_logy" if use_log_y else ""
    _save_paper_figure(
        fig,
        output_dir,
        f"prefix_median_packet_byte_comparison{suffix}",
        PAPER_FIGURE_FORMATS,
        PAPER_FIGURE_OVERWRITE,
    )
    plt.show()
    display(plot_df[[c for c in ["display_label", "prefix", "flow_count", "packet_count_median", "byte_count_median", "duration_median", "tcp_ratio", "udp_ratio"] if c in plot_df.columns]])


def plot_prefix_duration_median_comparison(
    overall_df: pd.DataFrame,
    prefix_df: pd.DataFrame,
    output_dir: Path,
    max_prefixes: int = 8,
    sort_by: str = "flow_count",
    use_log_y: bool = False,
) -> None:
    """Plot duration median separately because its unit differs from packet/byte counts."""
    if "duration_median" not in pd.concat([overall_df, prefix_df], ignore_index=True, sort=False).columns:
        print("[warn] duration_median column is not available; skip duration figure")
        return
    if overall_df.empty or prefix_df.empty:
        print("[warn] overall or prefix summary is empty; skip duration figure")
        return
    work_prefix = prefix_df.copy()
    if sort_by not in work_prefix.columns:
        sort_by = "flow_count" if "flow_count" in work_prefix.columns else "duration_median"
    work_prefix[sort_by] = pd.to_numeric(work_prefix[sort_by], errors="coerce")
    work_prefix = work_prefix.sort_values(sort_by, ascending=False).head(max_prefixes)

    overall_row = overall_df.iloc[[0]].copy()
    if "prefix" not in overall_row.columns:
        overall_row["prefix"] = "overall"
    overall_row["display_label"] = "overall"
    plot_df = pd.concat([overall_row, work_prefix], ignore_index=True, sort=False)
    plot_df["display_label"] = plot_df["display_label"].fillna(plot_df.get("prefix", "").astype(str).map(normalize_prefix_label))
    values = pd.to_numeric(plot_df["duration_median"], errors="coerce")

    fig, ax = plt.subplots(figsize=(7.0, 2.8))
    colors = ["#5B5B5B"] + ["#4C78A8"] * (len(plot_df) - 1)
    x = np.arange(len(plot_df))
    ax.bar(x, values, color=colors, edgecolor="white", linewidth=0.6)
    ax.set_ylabel("Median flow duration [s]")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["display_label"], rotation=35, ha="right")
    if use_log_y:
        positive = values[values > 0]
        if not positive.empty:
            ax.set_yscale("log")
            ax.set_ylabel("Median flow duration [s] (log scale)")
    fig.tight_layout()
    suffix = "_logy" if use_log_y else ""
    _save_paper_figure(
        fig,
        output_dir,
        f"prefix_median_duration_comparison{suffix}",
        PAPER_FIGURE_FORMATS,
        PAPER_FIGURE_OVERWRITE,
    )
    plt.show()


paper_overall_summary = load_overall_summary(PAPER_FIGURE_DATASET)
paper_prefix_summaries = load_prefix_summaries(PAPER_FIGURE_DATASET, min_flow_count=MIN_PREFIX_FLOW_COUNT)

print(f"paper figure dataset = {PAPER_FIGURE_DATASET}")
print(f"paper figure output directory = {PAPER_FIGURE_OUTPUT_DIR}")
print(f"prefix rows after flow_count >= {MIN_PREFIX_FLOW_COUNT}: {len(paper_prefix_summaries)}")

display_columns = [
    "display_label",
    "prefix",
    "flow_count",
    "packet_count_median",
    "byte_count_median",
    "duration_median",
    "avg_packet_size_median",
    "tcp_ratio",
    "udp_ratio",
]
display(paper_prefix_summaries[[c for c in display_columns if c in paper_prefix_summaries.columns]].head(MAX_PREFIXES_TO_PLOT * 2))

plot_prefix_median_comparison(
    paper_overall_summary,
    paper_prefix_summaries,
    PAPER_FIGURE_OUTPUT_DIR,
    max_prefixes=MAX_PREFIXES_TO_PLOT,
    sort_by=SORT_BY,
    use_log_y=False,
)

if PLOT_LOG_SCALE_VERSION:
    plot_prefix_median_comparison(
        paper_overall_summary,
        paper_prefix_summaries,
        PAPER_FIGURE_OUTPUT_DIR,
        max_prefixes=MAX_PREFIXES_TO_PLOT,
        sort_by=SORT_BY,
        use_log_y=True,
    )

plot_prefix_duration_median_comparison(
    paper_overall_summary,
    paper_prefix_summaries,
    PAPER_FIGURE_OUTPUT_DIR,
    max_prefixes=MAX_PREFIXES_TO_PLOT,
    sort_by=SORT_BY,
    use_log_y=False,
)
